In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

spark.conf.set("spark.sql.session.timeZone", "UTC")

station_schema = T.StructType([
    T.StructField("id", T.StringType()),
    T.StructField("name", T.StringType()),
    T.StructField(
        "geometry",
        T.StructType([
            T.StructField(
                "coordinates",
                T.ArrayType(T.DoubleType()),
            ),
        ]),
    ),
])

source_file_schema = T.StructType([
    T.StructField(
        "data",
        T.ArrayType(station_schema),
    ),
])

weather_sources = (
    spark.table("clubdata.bronze.weather_sources_raw")
    .withColumn(
        "source_document",
        F.from_json("raw_payload", source_file_schema),
    )
    .withColumn(
        "station",
        F.get(F.col("source_document.data"), 0),
    )
    .withColumn(
        "source_date",
        F.to_date(
            F.regexp_extract(
                "source_file",
                r"_(\d{4}-\d{2}-\d{2})_[0-9a-f]{12}\.json$",
                1,
            )
        ),
    )
    .select(
        "source_date",
        F.col("station.id").alias("weather_station_id"),
        F.col("station.name").alias("weather_station_name"),
        F.get(
            F.col("station.geometry.coordinates"), 1
        ).cast("double").alias("station_latitude"),
        F.get(
            F.col("station.geometry.coordinates"), 0
        ).cast("double").alias("station_longitude"),
    )
)

display(weather_sources)

In [0]:
measurement_schema = T.StructType([
    T.StructField("elementId", T.StringType()),
    T.StructField("value", T.DoubleType()),
    T.StructField("unit", T.StringType()),
    T.StructField("timeOffset", T.StringType()),
    T.StructField("timeResolution", T.StringType()),
    T.StructField("timeSeriesId", T.LongType()),
    T.StructField("qualityCode", T.IntegerType()),
])

observation_set_schema = T.StructType([
    T.StructField("sourceId", T.StringType()),
    T.StructField("referenceTime", T.StringType()),
    T.StructField(
        "observations",
        T.ArrayType(measurement_schema),
    ),
])

observation_file_schema = T.StructType([
    T.StructField(
        "data",
        T.ArrayType(observation_set_schema),
    ),
])

filename_pattern = (
    r"^observations_([^_]+)_"
    r"(\d{4}-\d{2}-\d{2}T\d{6}Z)_"
    r"[0-9a-f]{12}\.json$"
)

observation_files = (
    spark.table("clubdata.bronze.weather_observations_raw")
    .withColumn(
        "observation_document",
        F.from_json("raw_payload", observation_file_schema),
    )
    .withColumn(
        "weather_station_id",
        F.upper(
            F.regexp_extract("source_file", filename_pattern, 1)
        ),
    )
    .withColumn(
        "kickoff_token",
        F.regexp_extract("source_file", filename_pattern, 2),
    )
    .withColumn(
        "match_kickoff_at",
        F.to_timestamp(
            "kickoff_token",
            "yyyy-MM-dd'T'HHmmss'Z'",
        ),
    )
    .withColumn(
        "match_date",
        F.to_date("match_kickoff_at"),
    )
    .select(
        F.col("source_file").alias("observation_file"),
        "weather_station_id",
        "match_kickoff_at",
        "match_date",
        "observation_document",
    )
)

display(observation_files)

In [0]:
match_links = (
    spark.table("clubdata.silver.matches")
    .filter(F.col("venue_id").isNotNull())
    .select(
        "match_id",
        "venue_id",
        F.col("kickoff_at").alias("match_kickoff_at"),
        F.to_date("kickoff_at").alias("match_date"),
    )
)

venue_coordinates = (
    spark.table("clubdata.silver.venues")
    .select(
        "venue_id",
        F.col("latitude").alias("venue_latitude"),
        F.col("longitude").alias("venue_longitude"),
    )
)

weather_sources_for_join = (
    weather_sources
    .withColumnRenamed("source_date", "match_date")
)

linked_files = (
    observation_files
    .join(
        match_links,
        on=["match_kickoff_at", "match_date"],
        how="inner",
    )
    .join(
        weather_sources_for_join,
        on=["weather_station_id", "match_date"],
        how="inner",
    )
    .join(
        venue_coordinates,
        on="venue_id",
        how="left",
    )
)

print(f"Koblede observasjonsfiler: {linked_files.count()}")

In [0]:
observation_sets = (
    linked_files
    .select(
        "*",
        F.posexplode("observation_document.data").alias(
            "set_index",
            "observation_set",
        ),
    )
)

measurements = (
    observation_sets
    .select(
        "venue_id",
        "weather_station_id",
        "weather_station_name",
        "station_latitude",
        "station_longitude",
        "venue_latitude",
        "venue_longitude",
        "observation_file",
        "set_index",
        F.col("observation_set.referenceTime").alias(
            "reference_time_raw"
        ),
        F.posexplode("observation_set.observations").alias(
            "observation_index",
            "observation",
        ),
    )
    .select(
        "venue_id",
        "weather_station_id",
        "weather_station_name",
        "station_latitude",
        "station_longitude",
        "venue_latitude",
        "venue_longitude",
        "observation_file",
        F.to_timestamp("reference_time_raw").alias("reference_time"),
        F.col("observation.elementId").alias("element"),
        F.col("observation.value").cast("double").alias("value"),
        F.col("observation.unit").alias("unit"),
        F.col("observation.timeOffset").alias("time_offset"),
        F.col("observation.timeResolution").alias("time_resolution"),
        F.col("observation.timeSeriesId").alias("time_series_id"),
        F.col("observation.qualityCode").alias("quality_code"),
        (
            F.col("set_index") * 10000
            + F.col("observation_index")
        ).alias("source_order"),
    )
)

unsupported_offsets = [
    row.time_offset
    for row in (
        measurements
        .filter(
            F.col("time_offset").isNotNull()
            & ~F.col("time_offset").isin("PT0H", "PT20M")
        )
        .select("time_offset")
        .distinct()
        .collect()
    )
]

if unsupported_offsets:
    raise ValueError(
        f"Unsupported Frost time offsets: {unsupported_offsets}"
    )

measurements = (
    measurements
    .withColumn(
        "offset_minutes",
        F.when(
            F.col("time_offset").isNull()
            | (F.col("time_offset") == "PT0H"),
            0,
        )
        .when(F.col("time_offset") == "PT20M", 20)
        .cast("int"),
    )
    .withColumn(
        "observed_at",
        F.expr(
            "timestampadd(MINUTE, offset_minutes, reference_time)"
        ),
    )
)

print(f"Rå målinger etter utpakking: {measurements.count()}")

In [0]:
canonical_series = Window.partitionBy(
    "venue_id",
    "weather_station_id",
    "observed_at",
    "element",
).orderBy(
    F.asc(F.coalesce("quality_code", F.lit(999))),
    F.asc(
        F.when(F.col("time_resolution") == "PT1H", 0)
        .when(F.col("time_resolution") == "PT30M", 1)
        .when(F.col("time_resolution") == "PT10M", 2)
        .otherwise(3)
    ),
    F.asc(F.coalesce("time_series_id", F.lit(999))),
    F.asc("source_order"),
)

ranked_measurements = (
    measurements
    .withColumn(
        "canonical_rank",
        F.row_number().over(canonical_series),
    )
    .filter(F.col("canonical_rank") == 1)
)

latitude_a = F.radians("venue_latitude")
latitude_b = F.radians("station_latitude")
latitude_delta = latitude_b - latitude_a
longitude_delta = (
    F.radians("station_longitude")
    - F.radians("venue_longitude")
)

distance_component = (
    F.pow(F.sin(latitude_delta / 2), 2)
    + F.cos(latitude_a)
    * F.cos(latitude_b)
    * F.pow(F.sin(longitude_delta / 2), 2)
)

silver_weather = (
    ranked_measurements
    .withColumn(
        "distance_to_venue_km",
        F.lit(6371.0088)
        * 2
        * F.atan2(
            F.sqrt(distance_component),
            F.sqrt(1 - distance_component),
        ),
    )
    .select(
        "venue_id",
        "weather_station_id",
        "weather_station_name",
        "observed_at",
        "element",
        "value",
        "unit",
        "station_latitude",
        "station_longitude",
        "distance_to_venue_km",
        F.lit("Frost").alias("source"),
    )
)

In [0]:
(
    silver_weather.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("clubdata.silver.weather_observations")
)

print("Opprettet clubdata.silver.weather_observations")